# 🛡️ Transaction Fraud Scorer Agentic AI (Google Colab Setup & Run)

Welcome to the **Transaction Fraud Scorer Agentic AI** interactive Colab notebook!

This notebook demonstrates how to:
1. Clone the repository, install dependencies, and immediately launch the **FastAPI Gateway** (`src/api_gateway`) exposed via **Cloudflare Tunneling (`cloudflared`)**.
2. Install and run a local **Ollama** server directly inside Google Colab (CPU or free T4 GPU tier) and pull **`llama3.2:3b`**.
3. Set up **Model Context Protocol (MCP)** fraud scoring tools (Velocity, Geolocation Anomaly, Merchant Risk, Account History).
4. Orchestrate an autonomous AI Agent using **LangChain (`ChatOllama`)** that dynamically investigates transactions, executes MCP tools, and produces explainable risk assessments (`APPROVE`, `REVIEW`, `DECLINE`).

## Step 1: Verify Hardware Environment
Check if a GPU is assigned. If running on CPU, `llama3.2:3b` (and `llama3.2:1b`) will still run smoothly!

In [3]:
!nvidia-smi || echo "Running in CPU mode. Lightweight models like llama3.2:3b run well on CPU too!"

Sat Oct 31 18:52:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   35C    P0             87W /  600W |    2655MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Step 2: Clone Repository & Install Dependencies
Clone the project repository and install required packages (`fastapi`, `uvicorn`, `websockets`, `langchain`, `langchain-ollama`, `mcp`, `pydantic`, `rich`).

In [ ]:
import os
import sys

REPO_URL = "https://github.com/daknamsubho/TransactionFraudScorerAgenticAI.git"
REPO_DIR = "/content/TransactionFraudScorerAgenticAI" if os.path.exists("/content") else os.path.abspath("TransactionFraudScorerAgenticAI")
BRANCH = "feature/local_models"

# Clone or update repository on the target branch
if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    print(f"📥 Cloning {REPO_URL} (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"🔄 Repository found at {REPO_DIR}. Fetching latest changes on {BRANCH}...")
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

os.chdir(REPO_DIR)
print("📂 Working directory:", os.getcwd())

# Install project dependencies
!pip install -q "fastapi>=0.115.0" "uvicorn[standard]>=0.30.0" "websockets>=12.0"
!pip install -q "langchain>=1.4.0" "langchain-ollama>=1.1.0" "mcp>=1.3.0"
!pip install -q "pydantic>=2.7.0" "pydantic-settings>=2.2.0" "rich>=13.7.0" "httpx>=0.27.0"
!pip install -q -e .

src_path = os.path.join(REPO_DIR, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("✅ Environment and dependencies configured successfully!")

Not Found Repo. Clonning TransactionFraudScorerAgenticAI...
Cloning into 'TransactionFraudScorerAgenticAI'...
remote: Enumerating objects: 108, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 108 (delta 36), reused 90 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (108/108), 256.78 KiB | 11.16 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/TransactionFraudScorerAgenticAI/TransactionFraudScorerAgenticAI/TransactionFraudScorerAgenticAI/TransactionFraudScorerAgenticAI
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fraud-scorer (pyproject.toml) ... done
✅ Environment and dependencies configured successfully!


## Step 3: Start FastAPI Gateway & Expose via Cloudflare Tunnel (`cloudflared`)
Start the FastAPI API Gateway (`src/api_gateway`) on port `8000` in the background and expose it externally via a **Cloudflare Quick Tunnel** (`https://*.trycloudflare.com`).

Copy the printed URLs into your local website client to ingest transactions and stream real-time WebSocket score events!

In [11]:
import threading
import time
import httpx
import uvicorn

from api_gateway.app import app
from api_gateway.tunnel import start_cloudflare_tunnel

GATEWAY_PORT = 8000

def _run_uvicorn() -> None:
    uvicorn.run(app, host="0.0.0.0", port=GATEWAY_PORT, log_level="info")

# Start FastAPI server in a daemon thread if not already listening
try:
    httpx.get(f"http://127.0.0.1:{GATEWAY_PORT}/health", timeout=1.0)
    print("✅ FastAPI Gateway is already running on port", GATEWAY_PORT)
except Exception:
    server_thread = threading.Thread(target=_run_uvicorn, daemon=True)
    server_thread.start()
    time.sleep(2)
    print("✅ FastAPI Gateway started on http://127.0.0.1:8000")

# Start Cloudflare Quick Tunnel
tunnel_proc, public_url = start_cloudflare_tunnel(port=GATEWAY_PORT)
ws_url = public_url.replace("https://", "wss://") + "/ws/transactions/events"

print("\n" + "=" * 72)
print("🌐 CLOUDFLARE TUNNEL ACTIVE — USE THESE ENDPOINTS IN YOUR LOCAL WEBSITE")
print("=" * 72)
print(f"🔗 Public Base URL : {public_url}")
print(f"📖 Swagger UI Docs : {public_url}/docs")
print(f"📨 Score Endpoint  : POST {public_url}/api/v1/transactions/score")
print(f"⚖️ Decision API    : POST {public_url}/api/v1/transactions/decision")
print(f"📡 WebSocket Stream: {ws_url}")
print("=" * 72)

ModuleNotFoundError: No module named 'api_gateway'

## Step 4: Install Ollama in Colab & Launch Background Server
We install `zstd` (required by the Ollama installer) and Ollama using its official one-line script, then start the background daemon process on `http://localhost:11434`. Once running, the FastAPI Gateway automatically uses Ollama for live LLM evaluation.

In [ ]:
import subprocess
import time

import requests

print("🔧 Installing prerequisite zstd package...")
!apt-get update -qq && apt-get install -y -qq zstd

print("📥 Downloading and installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

print("🚀 Starting Ollama daemon in background...")
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# Check health
for i in range(10):
    try:
        resp = requests.get("http://localhost:11434/api/version")
        if resp.status_code == 200:
            print(f"✅ Ollama server is UP and ready! Version: {resp.json().get('version')}")
            break
    except Exception:
        time.sleep(1)
else:
    print("⚠️ Ollama is still initializing. Please wait a moment.")

## Step 5: Pull Open-Source Model (`llama3.2:3b`)
Download `llama3.2:3b` (~2.0 GB). It provides high reasoning quality and native tool calling.

In [ ]:
print("📦 Pulling llama3.2:3b model into Ollama...")
!ollama pull llama3.2:3b

print("\n📋 Available local models:")
!ollama list

## Step 6: Test MCP Risk Tools & LangChain Tool Adapters
Verify that the risk evaluation tools (Velocity, Geolocation, Merchant MCC, Account History) are properly loaded.

In [ ]:
from fraud_scorer.agent.tools import get_langchain_fraud_tools

tools = get_langchain_fraud_tools()
print(f"Loaded {len(tools)} LangChain MCP Tool Adapters:")
for t in tools:
    summary = t.description.strip().splitlines()[0]
    print(f"  • {t.name}: {summary}")

# Run a test invocation of velocity check
test_velocity = tools[0].invoke({"account_id": "acc_normal_01", "window_minutes": 60})
print("\nSample Tool Invocation Output:", test_velocity)

## Step 7: Initialize LangChain Agent Orchestrator with `ChatOllama`
Initialize `FraudScorerOrchestrator` pointing to the live Ollama daemon.

In [ ]:
from fraud_scorer.agent.orchestrator import FraudScorerOrchestrator, display_assessment, is_ollama_available
from fraud_scorer.models.transaction import GeoLocation, MerchantInfo, PaymentMethod, Transaction

print("Ollama server connected:", is_ollama_available("http://localhost:11434"))

# Initialize Orchestrator backed by ChatOllama
orchestrator = FraudScorerOrchestrator(
    ollama_base_url="http://localhost:11434", model_name="llama3.2:3b", provider="ollama"
)
print(f"✅ Fraud Scorer Orchestrator initialized using {orchestrator.model_name}!")

## Step 8: Scenario 1 - Normal Local Coffee Purchase
Evaluate a legitimate transaction ($42.50 in San Francisco at Blue Bottle Coffee by an established account).
Expected outcome: **`APPROVE`** with low composite risk score.

In [ ]:
tx_normal = Transaction(
    account_id="acc_normal_01",
    amount=42.50,
    currency="USD",
    location=GeoLocation(latitude=37.7749, longitude=-122.4194, city="San Francisco", country="US"),
    payment_method=PaymentMethod(payment_type="credit_card", card_last_four="4242", is_card_present=True),
    merchant=MerchantInfo(merchant_id="merch_cafe_88", name="Blue Bottle Coffee", mcc="5499"),
)

print("🔍 Evaluating Scenario 1 with LangChain & Ollama...")
assessment_1 = await orchestrator.evaluate_transaction(tx_normal)
display_assessment(tx_normal, assessment_1)

## Step 9: Scenario 2 - Multi-Vector Fraudulent Transaction
Evaluate a high-risk transaction ($3,500.00 in Tokyo at a casino shortly after London activity with high velocity and new untrusted account profile).
Expected outcome: **`DECLINE`** with triggered critical risk factors (impossible travel, gambling MCC, burst velocity).

In [ ]:
tx_suspicious = Transaction(
    account_id="acc_suspicious_01",
    amount=3500.00,
    currency="USD",
    location=GeoLocation(latitude=35.6762, longitude=139.6503, city="Tokyo", country="JP"),
    payment_method=PaymentMethod(payment_type="credit_card", card_last_four="9911", is_card_present=False),
    merchant=MerchantInfo(merchant_id="merch_casino_77", name="Tokyo High Roller Casino", mcc="7995"),
)

print("🔍 Evaluating Scenario 2 with LangChain & Ollama...")
assessment_2 = await orchestrator.evaluate_transaction(tx_suspicious)
display_assessment(tx_suspicious, assessment_2)

## Step 10: Interactive Custom Transaction Evaluator
You can test any custom scenario by modifying the transaction parameters below and re-running the cell!

In [ ]:
custom_tx = Transaction(
    account_id="acc_normal_01",
    amount=2400.00,
    currency="USD",
    location=GeoLocation(latitude=40.7128, longitude=-74.0060, city="New York", country="US"),
    payment_method=PaymentMethod(payment_type="credit_card", card_last_four="1234", is_card_present=False),
    merchant=MerchantInfo(merchant_id="merch_luxury_goods", name="Fifth Avenue Electronics", mcc="5732"),
)

print("🔍 Evaluating Custom Transaction...")
custom_assessment = await orchestrator.evaluate_transaction(custom_tx)
display_assessment(custom_tx, custom_assessment)

## Step 11: Verify End-to-End API Gateway (REST Score + WebSocket Event + Decision)
Verify that:
1. A WebSocket client connected to `/ws/transactions/events` receives the real-time event (`transaction_id`, `username`, `amount`, `score`, `score_details`).
2. `POST /api/v1/transactions/score` scores both **Card** and **ACH** transactions.
3. `POST /api/v1/transactions/decision` records an `Accept`/`Reject` decision with a `Reason`.

In [ ]:
import asyncio
import json
import httpx
import websockets

local_ws_url = f"ws://127.0.0.1:{GATEWAY_PORT}/ws/transactions/events"
local_http_url = f"http://127.0.0.1:{GATEWAY_PORT}"

sample_card_payload = {
    "username": "jdoe_99",
    "datetime": "2026-10-31T18:09:11Z",
    "ip": "198.51.100.42",
    "device_id": "dev_macbook_pro_91a",
    "location": {
        "latitude": 37.7749,
        "longitude": -122.4194,
        "city": "San Francisco",
        "state": "CA",
        "country": "US",
        "postal_code": "94103"
    },
    "transaction_details": {
        "transaction_type": "CARD",
        "amount": 1450.00,
        "currency": "USD",
        "merchant": {
            "merchant_id": "merch_88412",
            "name": "ElectroWorld Online",
            "mcc": "5732",
            "country": "US"
        },
        "card_details": {
            "card_type": "CREDIT",
            "card_network": "VISA",
            "card_last_four": "4242",
            "is_card_present": False,
            "expiration_month": 8,
            "expiration_year": 2028,
            "cvv_verified": True,
            "avs_result": "Y"
        }
    }
}

async with websockets.connect(local_ws_url) as ws:
    async with httpx.AsyncClient(timeout=60.0) as client:
        # 1. Score the transaction
        score_resp = await client.post(f"{local_http_url}/api/v1/transactions/score", json=sample_card_payload)
        score_data = score_resp.json()
        print("✅ POST /api/v1/transactions/score Response:")
        print(json.dumps(score_data, indent=2))

        # 2. Receive real-time WebSocket event
        ws_msg = json.loads(await asyncio.wait_for(ws.recv(), timeout=5.0))
        print("\n📡 WebSocket Event Received (/ws/transactions/events):")
        print(json.dumps(ws_msg, indent=2))

        # 3. Submit Accept/Reject decision
        decision_payload = {
            "transaction_id": score_data["transaction_id"],
            "accept": True,
            "reason": "Customer verified transaction via 2FA challenge."
        }
        decision_resp = await client.post(f"{local_http_url}/api/v1/transactions/decision", json=decision_payload)
        print("\n⚖️ POST /api/v1/transactions/decision Response:")
        print(json.dumps(decision_resp.json(), indent=2))